In [2]:
import tensorflow as tf
print(tf.__version__)

2.20.0


In [3]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="3paRDUGoYZN4W5hH8RPH")
project = rf.workspace("shashika-sandaruwan").project("plastic-waste-classification-aazyj-hqjnv")
version = project.version(1)
dataset = version.download("folder")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.9/207.9 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 40.9 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.13
    Uninstalling idna-3.13:
      Successfully uninstalled idna-3.13


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Plastic-Waste-Classification-1 in folder:: 100%|██████████| 397/397 [00:00<00:00, 1313.14it/s]


In [4]:
import tensorflow as tf
from tensorflow.keras.preprocessing import image_dataset_from_directory


DATA_PATH = '/content/Plastic-Waste-Classification-1/train'

train_ds = image_dataset_from_directory(
    DATA_PATH,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(224, 224),
    batch_size=32)

val_ds = image_dataset_from_directory(
    DATA_PATH,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(224, 224),
    batch_size=32)

class_names = train_ds.class_names
print("සාර්ථකයි! හඳුනාගත් වර්ග:", class_names)

Found 316 files belonging to 6 classes.
Using 253 files for training.
Found 316 files belonging to 6 classes.
Using 63 files for validation.
සාර්ථකයි! හඳුනාගත් වර්ග: ['1-PET', '2-HDPE', '3-PVC', '4-LDPE', '5-PP', '6-PS']


In [5]:

base_model = tf.keras.applications.MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
base_model.trainable = False

model = tf.keras.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(len(class_names), activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Model එක සාර්ථකව නිර්මාණය කළා!


In [6]:
history = model.fit(train_ds, validation_data=val_ds, epochs=10)

Epoch 1/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.2482 - loss: 2.1400

KeyboardInterrupt: 

In [1]:
import gradio as gr
import tensorflow as tf
import numpy as np

def predict_plastic(img):
    try:

        img_array = tf.image.resize(img, [224, 224])
        img_array = tf.keras.applications.mobilenet_v2.preprocess_input(img_array)
        img_array = np.expand_dims(img_array, axis=0)


        prediction = model.predict(img_array, verbose=0)[0]
        predicted_class = class_names[np.argmax(prediction)]

        return f"Result: {predicted_class}"
    except Exception as e:
        return f"Error: {str(e)}"


demo = gr.Interface(
    fn=predict_plastic,
    inputs=gr.Image(type="numpy"),
    outputs="text",
    title="Plastic Classifier"
)

demo.launch(share=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>